# 01event: download the Choi-wan event data

Part A of PF5 (direct export): the haze streamer from eastern Indonesia into Super Typhoon Choi-wan.

This notebook only downloads. Maps, haze outlines, trajectories and smoke export come in the next notebooks. All files go to `data/raw/event/`. Each step skips files that already exist, so the notebook can be re-run after an interruption without downloading everything again.

| Section | Source | What |
|---|---|---|
| 2 | GEOS-FP (NASA, OPeNDAP) | aerosol optical depth, PM2.5 components, smoke column flux, wet deposition, 3-hourly |
| 3 | CAMS (Copernicus ADS) | PM2.5, PM10, aerosol optical depth, 3-hourly |
| 4 | ERA5 single levels (Copernicus CDS) | 10 m wind, rain, sea level pressure, hourly |
| 5 | ERA5 pressure levels (Copernicus CDS) | wind at 850, 700 and 500 hPa, 3-hourly (for trajectories) |
| 6 | NASA FIRMS | fire hotspots from VIIRS (three satellites) and MODIS, daily |
| 7 | IBTrACS (NOAA) | Choi-wan best track |

Expected run time: about 15 to 40 minutes, mostly GEOS-FP and the Copernicus queues.

## Section 0: settings

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
KEYS_FILE = PROJECT_DIR / "config" / "keys.toml"
OUT_DIR = PROJECT_DIR / "data" / "raw" / "event"

EVENT_START = "2026-09-28"  # <- first day of the event window
EVENT_END = "2026-10-07"  # <- last day of the event window (included)

# Study box in degrees (east positive, north positive)
BOX_WEST = 90.0  # <-
BOX_EAST = 160.0  # <-
BOX_SOUTH = -15.0  # <-
BOX_NORTH = 45.0  # <-

# Steps to run (set False to skip one)
DO_GEOSFP = True  # <-
DO_CAMS = True  # <-
DO_ERA5_SINGLE = True  # <-
DO_ERA5_LEVELS = True  # <-
DO_FIRMS = True  # <-
DO_TRACK = True  # <-

ERA5_LEVELS = ["850", "700", "500"]  # <- pressure levels in hPa
FIRMS_SOURCES = ["VIIRS_SNPP_NRT", "VIIRS_NOAA20_NRT", "VIIRS_NOAA21_NRT", "MODIS_NRT"]  # <-
STORM_NAME = "CHOI-WAN"  # <- as written in IBTrACS
STORM_SEASON = 2026  # <-

## Section 1: imports, folders, keys and helpers

In [2]:
import io
import time
import shutil
import tomllib
import zipfile
import requests
import numpy as np
import pandas as pd
import xarray as xr
import cdsapi
import warnings

warnings.filterwarnings("ignore", message="Ambiguous reference date string")
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(KEYS_FILE, "rb") as f:
    KEYS = tomllib.load(f)

DAYS = pd.date_range(EVENT_START, EVENT_END, freq="D")
AREA = [BOX_NORTH, BOX_WEST, BOX_SOUTH, BOX_EAST]
RESULTS = {}
print("Event window:", DAYS[0].date(), "to", DAYS[-1].date(), f"({len(DAYS)} days)")
print("Output folder:", OUT_DIR)

Event window: 2026-09-28 to 2026-10-07 (10 days)
Output folder: .\data\raw\event


In [3]:
def done(path):
    # A file counts as done if it exists and is not empty
    return path.exists() and path.stat().st_size > 0


def open_geosfp(url):
    # netCDF4 first (fast), pydap second (works on Windows)
    for engine in ["netcdf4", "pydap"]:
        try:
            target = url.replace("https://", "dap2://") if engine == "pydap" else url
            ds = xr.open_dataset(target, engine=engine)
            print("GEOS-FP opened with", engine)
            return ds
        except Exception as e:
            print(f"GEOS-FP engine {engine} failed:", type(e).__name__)
    raise OSError("no engine could open GEOS-FP")


def era5_by_month(dataset, request, prefix):
    # One request per calendar month, so only the window's days are asked for
    cds = cdsapi.Client(url=KEYS["cds"]["url"], key=KEYS["cds"]["key"], quiet=True, progress=False, retry_max=5, sleep_max=30)
    written = []
    for month, days in DAYS.to_series().groupby(DAYS.strftime("%Y-%m")):
        out_file = OUT_DIR / f"{prefix}_{month}.nc"
        existing = [f for f in OUT_DIR.glob(f"{prefix}_{month}*.nc") if done(f)]
        if existing:
            print(f"{prefix}_{month}: already there ({len(existing)} file(s)), skipped")
            written += existing
            continue
        req = dict(request)
        req["year"] = [month[:4]]
        req["month"] = [month[5:]]
        req["day"] = list(days.dt.strftime("%d"))
        t0 = time.time()
        tmp = out_file.with_suffix(".download")
        cds.retrieve(dataset, req, str(tmp))
        written += unzip_netcdf(tmp, out_file)
        tmp.unlink()
        print(f"{out_file.name}: {len(req['day'])} days in {time.time() - t0:.0f} s")
    return written


def unzip_netcdf(zip_path, out_path):
    # Copernicus sometimes delivers a zip with one or more .nc files inside
    if not zipfile.is_zipfile(zip_path):
        shutil.copy(zip_path, out_path)
        return [out_path]
    written = []
    with zipfile.ZipFile(zip_path) as z:
        names = [n for n in z.namelist() if n.endswith(".nc")]
        for n in names:
            target = out_path if len(names) == 1 else out_path.with_name(out_path.stem + "_" + Path(n).stem + ".nc")
            with z.open(n) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
            written.append(target)
    return written

## Section 2: GEOS-FP aerosol (NASA)

GEOS-FP is the model behind earth.nullschool.net. The collection `tavg3_2d_aer_Nx` holds 3-hour means, stamped at 01:30, 04:30, and so on.

The cell first saves the full variable list with descriptions to `geosfp_variables.csv` (useful for the next notebooks), then downloads each wanted variable for the event window and box into its own file. Variables that this GEOS-FP version does not have are skipped and reported.

- `totexttau`: total aerosol optical depth (the "how thick is the haze" number)
- `pm25`: NASA's own combined surface PM2.5 (the number nullschool shows)
- `dusmass25`, `sssmass25`, `bcsmass`, `ocsmass`, `brsmass`, `so4smass`, `nismass25`, `nh4smass`: surface mass of the PM2.5 parts (dust, sea salt, black carbon, organic carbon, brown carbon, sulfate, nitrate, ammonium), to see what the haze is made of
- `bcfluxu`, `bcfluxv`, `ocfluxu`, `ocfluxv`, `brfluxu`, `brfluxv`: column mass flux of smoke east and north (how much smoke crosses a line per second), used for the export estimate in notebook 04
- `bccmass`, `occmass`, `brcmass`: total smoke in the whole air column above each point
- names starting with `bcwt` and `ocwt`: wet deposition of smoke, if this collection has it (it does not at present; notebook 04 will look elsewhere)

In [4]:
GEOSFP_URL = "https://opendap.nccs.nasa.gov/dods/GEOS-5/fp/0.25_deg/assim/tavg3_2d_aer_Nx"
GEOSFP_WANTED = ["totexttau", "pm25", "dusmass25", "sssmass25", "bcsmass", "ocsmass", "brsmass", "so4smass", "nismass25", "nh4smass", "bcfluxu", "bcfluxv", "ocfluxu", "ocfluxv", "brfluxu", "brfluxv", "bccmass", "occmass", "brcmass"]
GEOSFP_PREFIXES = ["bcwt", "ocwt"]
GEOSFP_TRIES = 3  # <- attempts per variable before giving up

if DO_GEOSFP:
    try:
        geos = open_geosfp(GEOSFP_URL)
        var_table = pd.DataFrame([(v, geos[v].attrs.get("long_name", ""), geos[v].attrs.get("units", "")) for v in geos.data_vars], columns=["variable", "long_name", "units"])
        var_table.to_csv(OUT_DIR / "geosfp_variables.csv", index=False)
        print("Variables in this GEOS-FP collection:", len(var_table))
        available = set(geos.data_vars)
        wanted = [v for v in GEOSFP_WANTED if v in available] + sorted(v for v in available if any(v.startswith(p) for p in GEOSFP_PREFIXES))
        missing = [v for v in GEOSFP_WANTED if v not in available]
        print("Will download:", wanted)
        print("Not in this collection:", missing)
        geos_times = pd.to_datetime([str(t) for t in geos.time.values])
        in_window = np.where((geos_times >= DAYS[0]) & (geos_times < DAYS[-1] + pd.Timedelta(days=1)))[0]
        print("Time steps in window:", len(in_window), "from", geos_times[in_window[0]], "to", geos_times[in_window[-1]])
        failed = []
        for v in wanted:
            out_file = OUT_DIR / f"geosfp_{v}.nc"
            if done(out_file):
                print(f"{v}: already there, skipped")
                continue
            for attempt in range(1, GEOSFP_TRIES + 1):
                try:
                    t0 = time.time()
                    part = geos[v].isel(time=slice(int(in_window[0]), int(in_window[-1]) + 1)).sel(lat=slice(BOX_SOUTH, BOX_NORTH), lon=slice(BOX_WEST, BOX_EAST)).load()
                    part.astype("float32").to_netcdf(out_file)
                    print(f"{v}: {part.shape} in {time.time() - t0:.0f} s")
                    break
                except Exception as e:
                    print(f"{v}: attempt {attempt} failed:", type(e).__name__, str(e)[:150])
                    time.sleep(10)
            else:
                failed.append(v)
        RESULTS["GEOS-FP"] = "OK" if not failed else f"PARTLY ({len(failed)} failed: {', '.join(failed)})"
    except Exception as e:
        print("GEOS-FP step failed:", type(e).__name__, e)
        RESULTS["GEOS-FP"] = f"FAIL ({type(e).__name__})"

GEOS-FP engine netcdf4 failed: OSError


.\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GEOS-FP opened with pydap
Variables in this GEOS-FP collection: 91
Will download: ['totexttau', 'pm25', 'dusmass25', 'sssmass25', 'bcsmass', 'ocsmass', 'brsmass', 'so4smass', 'nismass25', 'nh4smass', 'bcfluxu', 'bcfluxv', 'ocfluxu', 'ocfluxv', 'brfluxu', 'brfluxv', 'bccmass', 'occmass', 'brcmass']
Not in this collection: []
Time steps in window: 80 from 2026-09-28 01:30:00 to 2026-10-07 22:30:00
totexttau: already there, skipped
pm25: (80, 241, 225) in 53 s
dusmass25: already there, skipped
sssmass25: (80, 241, 225) in 128 s
bcsmass: already there, skipped
ocsmass: already there, skipped
brsmass: (80, 241, 225) in 111 s
so4smass: already there, skipped
nismass25: already there, skipped
nh4smass: attempt 1 failed: ChunkedEncodingError ('Connection broken: IncompleteRead(0 bytes read, 8192 more expected)', IncompleteRead(0 bytes read, 8192 more expected))
nh4smass: (80, 241, 225) in 85 s
bcfluxu: already there, skipped
bcfluxv: already there, skipped
ocfluxu: already there, skipped
ocflu

## Section 3: CAMS (Copernicus Atmosphere Data Store)

The CAMS analysis runs twice a day (00 and 12 UTC). Taking lead times 0, 3, 6 and 9 hours from each run gives a 3-hourly series. CAMS stores PM2.5 and PM10 in kg per cubic metre (times 1e9 gives micrograms per cubic metre).

In [5]:
cams_zip = OUT_DIR / "cams_event.zip"
cams_file = OUT_DIR / "cams_event.nc"
cams_request = {}
cams_request["variable"] = ["particulate_matter_2.5um", "particulate_matter_10um", "total_aerosol_optical_depth_550nm"]
cams_request["date"] = [f"{EVENT_START}/{EVENT_END}"]
cams_request["time"] = ["00:00", "12:00"]
cams_request["leadtime_hour"] = ["0", "3", "6", "9"]
cams_request["type"] = ["forecast"]
cams_request["data_format"] = "netcdf_zip"
cams_request["area"] = AREA

if DO_CAMS:
    try:
        if done(cams_file):
            print("CAMS: already there, skipped")
        else:
            ads = cdsapi.Client(url=KEYS["ads"]["url"], key=KEYS["ads"]["key"], quiet=True, progress=False, retry_max=5, sleep_max=30)
            t0 = time.time()
            ads.retrieve("cams-global-atmospheric-composition-forecasts", cams_request, str(cams_zip))
            written = unzip_netcdf(cams_zip, cams_file)
            print(f"CAMS: {[w.name for w in written]} in {time.time() - t0:.0f} s")
        RESULTS["CAMS"] = "OK"
    except Exception as e:
        print("CAMS step failed:", type(e).__name__, e)
        RESULTS["CAMS"] = f"FAIL ({type(e).__name__})"

CAMS: already there, skipped


## Section 4: ERA5 single levels (Copernicus Climate Data Store)

Hourly, because rain is stored as hourly totals: 3-hourly samples would miss most of it. One request per calendar month (files `era5_single_2026-09.nc`, `era5_single_2026-10.nc`), so only the days of the window are asked for.

In [6]:
era5s_request = {}
era5s_request["product_type"] = ["reanalysis"]
era5s_request["variable"] = ["10m_u_component_of_wind", "10m_v_component_of_wind", "total_precipitation", "mean_sea_level_pressure"]
era5s_request["time"] = [f"{h:02d}:00" for h in range(24)]
era5s_request["data_format"] = "netcdf"
era5s_request["download_format"] = "unarchived"
era5s_request["area"] = AREA

if DO_ERA5_SINGLE:
    try:
        era5_by_month("reanalysis-era5-single-levels", era5s_request, "era5_single")
        RESULTS["ERA5 single levels"] = "OK"
    except Exception as e:
        print("ERA5 single levels step failed:", type(e).__name__, e)
        RESULTS["ERA5 single levels"] = f"FAIL ({type(e).__name__})"

era5_single_2026-09: already there (2 file(s)), skipped
era5_single_2026-10: already there (2 file(s)), skipped


## Section 5: ERA5 pressure levels (Copernicus Climate Data Store)

Wind at 850 hPa (about 1.5 km up) carries most of the smoke in the lower atmosphere; 700 and 500 hPa show what happens higher up. These drive the trajectories in notebook 03.

Again one request per calendar month (`era5_levels_2026-09.nc`, `era5_levels_2026-10.nc`).

In [7]:
era5p_request = {}
era5p_request["product_type"] = ["reanalysis"]
era5p_request["variable"] = ["u_component_of_wind", "v_component_of_wind", "vertical_velocity"]
era5p_request["pressure_level"] = ERA5_LEVELS
era5p_request["time"] = [f"{h:02d}:00" for h in range(0, 24, 3)]
era5p_request["data_format"] = "netcdf"
era5p_request["download_format"] = "unarchived"
era5p_request["area"] = AREA

if DO_ERA5_LEVELS:
    try:
        era5_by_month("reanalysis-era5-pressure-levels", era5p_request, "era5_levels")
        RESULTS["ERA5 pressure levels"] = "OK"
    except Exception as e:
        print("ERA5 pressure levels step failed:", type(e).__name__, e)
        RESULTS["ERA5 pressure levels"] = f"FAIL ({type(e).__name__})"

era5_levels_2026-09: already there (1 file(s)), skipped
era5_levels_2026-10: already there (1 file(s)), skipped


## Section 6: NASA FIRMS fire hotspots

One request per satellite per day (the API allows a few days per request, but one day at a time keeps each answer small and easy to retry). Error messages never show the address, because it contains the key.

In [8]:
firms_file = OUT_DIR / "firms_event.csv"
FIRMS_KEY = KEYS["firms"]["key"]
box_text = f"{BOX_WEST:g},{BOX_SOUTH:g},{BOX_EAST:g},{BOX_NORTH:g}"

if DO_FIRMS:
    try:
        if done(firms_file):
            print("FIRMS: already there, skipped")
        else:
            parts = []
            for source in FIRMS_SOURCES:
                counts = []
                for day in DAYS:
                    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_KEY}/{source}/{box_text}/1/{day.date()}"
                    r = requests.get(url, timeout=120)
                    if r.status_code != 200:
                        print(f"{source} {day.date()}: HTTP {r.status_code}")
                        counts.append(0)
                        continue
                    df = pd.read_csv(io.StringIO(r.text))
                    if "latitude" not in df.columns:
                        print(f"{source} {day.date()}: message instead of data:", r.text[:120].replace(FIRMS_KEY, "***"))
                        counts.append(0)
                        continue
                    df["source"] = source
                    parts.append(df)
                    counts.append(len(df))
                print(f"{source}: {sum(counts)} hotspots, per day {counts}")
            fires = pd.concat(parts, ignore_index=True)
            fires["confidence"] = fires["confidence"].astype(str)
            fires.to_csv(firms_file, index=False)
            print("FIRMS total:", len(fires))
        RESULTS["FIRMS"] = "OK"
    except Exception as e:
        print("FIRMS step failed:", type(e).__name__, str(e).replace(FIRMS_KEY, "***"))
        RESULTS["FIRMS"] = f"FAIL ({type(e).__name__})"

VIIRS_SNPP_NRT: 130568 hotspots, per day [10718, 9919, 9229, 12290, 12322, 8313, 12633, 17420, 19615, 18109]
VIIRS_NOAA20_NRT: 139844 hotspots, per day [10220, 12550, 8868, 12068, 13809, 13671, 14065, 15010, 18743, 20840]
VIIRS_NOAA21_NRT: 134271 hotspots, per day [10320, 10349, 9061, 11908, 11598, 11671, 14175, 16683, 17349, 21157]
MODIS_NRT: 25390 hotspots, per day [1424, 2845, 1697, 2049, 3186, 2144, 2577, 3035, 2095, 4338]
FIRMS total: 430073


## Section 7: Choi-wan track from IBTrACS

IBTrACS (NOAA) merges the tracks of all warning centres. The "last 3 years" file includes recent storms as provisional data. If Choi-wan is not in it yet, the cell says so and we use another source.

The track also tells us whether the event window in Section 0 covers the storm's whole life.

In [9]:
IBTRACS_URL = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.last3years.list.v04r01.csv"
ibtracs_file = OUT_DIR / "ibtracs_last3years.csv"
track_file = OUT_DIR / "choiwan_track.csv"

if DO_TRACK:
    try:
        if not done(ibtracs_file):
            r = requests.get(IBTRACS_URL, timeout=300)
            print("IBTrACS download, HTTP status:", r.status_code)
            ibtracs_file.write_bytes(r.content)
        ib = pd.read_csv(ibtracs_file, skiprows=[1], low_memory=False)
        storm = ib[(ib["NAME"] == STORM_NAME) & (ib["SEASON"] == STORM_SEASON)].copy()
        print("Track points found:", len(storm))
        if len(storm) == 0:
            print("Not in IBTrACS yet. Latest storms in the file:")
            print(ib.groupby(["SEASON", "NAME"])["ISO_TIME"].max().sort_values().tail(5))
            RESULTS["IBTrACS track"] = "NOT YET IN IBTRACS"
        else:
            keep = [c for c in ["SID", "NAME", "ISO_TIME", "NATURE", "LAT", "LON", "WMO_WIND", "WMO_PRES", "USA_WIND", "USA_PRES", "USA_SSHS", "TOKYO_WIND", "TOKYO_PRES", "STORM_SPEED", "STORM_DIR"] if c in storm.columns]
            storm = storm[keep]
            storm.to_csv(track_file, index=False)
            print("First point:", storm["ISO_TIME"].iloc[0], " Last point:", storm["ISO_TIME"].iloc[-1])
            print(storm[["ISO_TIME", "LAT", "LON", "USA_WIND", "USA_PRES"]].iloc[::8].to_string(index=False))
            RESULTS["IBTrACS track"] = "OK"
    except Exception as e:
        print("Track step failed:", type(e).__name__, e)
        RESULTS["IBTrACS track"] = f"FAIL ({type(e).__name__})"

IBTrACS download, HTTP status: 200
Track points found: 59
First point: 2026-09-30 00:00:00  Last point: 2026-10-07 06:00:00
           ISO_TIME  LAT   LON USA_WIND USA_PRES
2026-09-30 00:00:00 15.6 151.9       25     1005
2026-10-01 00:00:00 16.7 147.9       39      997
2026-10-02 00:00:00 17.0 144.7       54      992
2026-10-03 00:00:00 18.6 145.3       84      969
2026-10-04 00:00:00 22.1 147.1      109      942
2026-10-05 00:00:00 25.2 146.5      128      916
2026-10-06 00:00:00 29.8 145.8       93      945
2026-10-07 00:00:00 39.3 152.6       72      962


## Section 8: summary

In [10]:
summary = pd.DataFrame(list(RESULTS.items()), columns=["Step", "Result"])
print(summary.to_string(index=False))
print()
files = sorted(OUT_DIR.glob("*"))
for p in files:
    print(f"{p.name:32s} {p.stat().st_size / 1e6:8.1f} MB")
print(f"Total: {sum(p.stat().st_size for p in files) / 1e6:.0f} MB")

                Step Result
             GEOS-FP     OK
                CAMS     OK
  ERA5 single levels     OK
ERA5 pressure levels     OK
               FIRMS     OK
       IBTrACS track     OK

cams_event.nc                        17.1 MB
cams_event.zip                       17.1 MB
choiwan_track.csv                     0.0 MB
era5_levels_2026-09.nc               30.8 MB
era5_levels_2026-10.nc               71.9 MB
era5_single_2026-09_data_stream-oper_stepType-accum.nc      5.2 MB
era5_single_2026-09_data_stream-oper_stepType-instant.nc     28.0 MB
era5_single_2026-10_data_stream-oper_stepType-accum.nc     11.6 MB
era5_single_2026-10_data_stream-oper_stepType-instant.nc     65.3 MB
firms_event.csv                      44.0 MB
geosfp_bccmass.nc                    17.4 MB
geosfp_bcfluxu.nc                    17.4 MB
geosfp_bcfluxv.nc                    17.4 MB
geosfp_bcsmass.nc                    17.4 MB
geosfp_brcmass.nc                    17.4 MB
geosfp_brfluxu.nc                   

___
**END**